# Preprocessing Dataset Ember Electricity

**Hal-hal yang dilakukan**
1. Load Data
2. Describe Data
3. Merapikan Nama Kolom
4. Perbaiki Tipe Data
5. Menambah Kolom Waktu
6. Membersihkan Data Kosong
7. Menghapus Data Duplikat
8. Validasi Struktur Hierarki
9. Penandaan Data Provisional
10. Pemisahan Data
11. Analisis Outlier pada Kolom Value
12. Ringkasan & Simpan Hasil

# Setup-Import Library

backup agar data original tidak berubah selama preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

print("Libraries imported successfully.")

# 1. Load Data

In [ ]:
FILE_PATH = 'monthly_full_release_long_format.csv'
df = pd.read_csv(FILE_PATH)
df_asli = df.copy()

print("Data loaded successfully.")
print(f'Jumlah Baris : {df.shape[0]:,}')
print(f'Jumlah Kolom : {df.shape[1]:,}')

# 2. Describe Data
Agar tahu kolom mana yang perlu diperbaiki, misalnya kolom flag (EU, G20, dll.) yang terbaca sebagai float64 padahal seharusnya integer

In [ ]:
print("5 baris pertama")
df.head()

In [ ]:
print("Info kolom dan tipe data")
df.info()

In [ ]:
#cek kolom yang kosong
kosong = df.isnull().sum()
persen = (kosong / len(df) * 100).round(1)

hasil_kosong = pd.DataFrame({
    'Jumlah Kosong': kosong,
    'Persentase Kosong (%)': persen
})

print("kolom yang punya data kosong")
print(hasil_kosong[hasil_kosong['Jumlah Kosong'] > 0])

In [ ]:
#cek nilai-nilai unik di kolom penting
for kolom in ['Category', 'Subcategory', 'Area type', 'Unit']:
    print(f"Nilai unik di kolom '[{kolom}]':")
    print(' ',df[kolom].unique())
    print()

#statistik dasar kolom angka
print("Statistik kolom angka")
df.describe()

# 3. Merapikan Nama Kolom
- Mengubah nama menjadi snake_case (huruf kecil semua dan menggunakan underscore) mempermudah penulisan kode di tahap berikutnya.

In [ ]:
print("Nama kolom sebelum dirapihkan:")
print(df.columns.tolist())

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('%', 'pct', regex=False)
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

print("Nama kolom setelah dirapihkan:")
print(df.columns.tolist())

# 4. Memperbaiki Tipe Data
- mengubah kolom kategori(Category, Sub Category, dan Unit) menjadi category untuk menghemat memori
- mengubah kolom bendera organisasi(EU, OECD, G20, G7, ASEAN) menjadi int8(bool) karena hanya bernilai 0 dan 1

In [ ]:
# cek tipe data dan contoh nilai untuk setiap kolom
print("Tipe data setiap kolom:")
tipe_data = pd.DataFrame({
    'Tipe Data': df.dtypes,
    'Contoh Nilai': [df[col].dropna().iloc[0] if df[col].dropna().shape[0] > 0 else 'N/A' 
    for col in df.columns]
})
print(tipe_data)

## 4.1 standarisasi tipe data

In [ ]:
# CEK DAN PERBAIKI KOLOM FLAG ORGANISASI
# Kolom-kolom ini seharusnya berisi hanya angka 0 atau 1

kolom_flag = ['eu', 'oecd', 'g20', 'g7', 'asean']

print("sebelum konversi:")
for kolom in kolom_flag:
    if kolom in df.columns:
        print(f" {kolom}: tipe data = {df[kolom].dtype}, NaN = {df[kolom].isnull().sum():,}")

# Ubah kolom flag ke tipe integer (kalau belum)
for kolom in kolom_flag:
    if kolom in df.columns:
        df[kolom] = df[kolom].fillna(0).astype('Int8')

print("\nSetelah konversi:")
for kolom in kolom_flag:
    if kolom in df.columns:
        print(f" {kolom}: tipe data = {df[kolom].dtype}, NaN = {df[kolom].isnull().sum():,}")

In [ ]:
kolom_cat = ['area', 'area_type', 'continent', 'ember_region', 'category', 'subcategory', 'variable', 'unit']
for kolom in kolom_cat:
    if kolom in df.columns:
        df[kolom] = df[kolom].astype('category')

tipe_data_setelah = pd.DataFrame({
    'Tipe Data': df.dtypes,
    'Contoh Nilai': [df[col].dropna().iloc[0] if df[col].dropna().shape[0] > 0 else 'N/A' 
    for col in df.columns]
})
print("\nTipe data setelah konversi:")
print(tipe_data_setelah)

## 4.2 isi kolom identitas yang kosong (ISO, Continent, Ember Region)

In [ ]:
kolom_identitas = ['iso_3_code', 'continent', 'ember_region']

print('Missing SEBELUM diisi:')
for col in kolom_identitas:
    if col in df.columns:
        print(f'  {col}: {df[col].isnull().sum():,} kosong')

# Isi dengan 'N/A' — Not Applicable, bukan data hilang
for col in kolom_identitas:
    if col in df.columns:
        if hasattr(df[col], 'cat') and 'N/A' not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories('N/A')
        df[col] = df[col].fillna('N/A')

print('\nMissing SESUDAH diisi:')
for col in kolom_identitas:
    if col in df.columns:
        print(f'  {col}: {df[col].isnull().sum():,} kosong')

# Sanity check: memastikan tidak ada negara yang ISO-nya ikut N/A
if 'area_type' in df.columns and 'iso_3_code' in df.columns:
    salah = df[df['area_type'] == 'Country or economy']['iso_3_code'].isin(['N/A']).sum()
    print(f'\nSanity check — Country dengan ISO = N/A: {salah} (harusnya 0)')

# 5. Menambah Turunan Kolom Waktu
- Kolom string YYYY-MM-01 sulit diolah untuk perhitungan. Dengan tipe datetime, bisa melakukan filter berdasarkan rentang waktu dengan mudah.
- Memisahkan Year mempermudah agregat tahunan tanpa harus memanipulasi string berulang kali.

In [ ]:
print("Tipe kolom date sebelum: ", df['date'].dtype)

df['date'] = pd.to_datetime(df['date'], errors='coerce')
print("Tipe kolom date setelah: ", df['date'].dtype)

#membuat kolom turunan dari date
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['quarter'] = df['date'].dt.quarter
df['yearmonth'] = df['date'].dt.to_period('M')

print("Kolom waktu baru yang berhasil dibuat: ")
df[['date', 'year', 'month', 'month_name', 'quarter', 'yearmonth']].drop_duplicates().head(12)

In [ ]:
print(f'Data tersedia dari  : {df["date"].min().date()}')
print(f'Data tersedia sampai: {df["date"].max().date()}')
print(f'Total tahun         : {df["year"].nunique()} tahun ({int(df["year"].min())} - {int(df["year"].max())})')

In [ ]:
# Visualisasi: Berapa banyak data per tahun?
# untuk lihat apakah ada tahun yang datanya sedikit/tidak lengkap

data_per_tahun = df.groupby('year').size()

plt.figure(figsize=(12, 4))
data_per_tahun.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Jumlah Baris Data per Tahun', fontsize=14, fontweight='bold')
plt.xlabel('Tahun')
plt.ylabel('Jumlah Baris')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nTotal rentang tahun: {df['year'].min()} - {df['year'].max()}")

# 6. Membersihkan data kosong
- Kolom value memiliki 3.292 baris kosong (0.7%) yang tidak bisa diimputasi karena tidak diketahui nilai sebenarnya, sehingga lebih aman dihapus
- Kolom YoY dibiarkan karena memang by design kosong di awal periode (37–46% kosong adalah wajar)

In [ ]:
print(f"total baris sebelum dibersihkan: {len(df):,}")

In [ ]:
# Cek missing value di kolom value per area_type
if 'area_type' in df.columns:
    print("Missing values di kolom 'value' per Area Type:")
    missing_per_type = df.groupby('area_type')['value'].apply(lambda x: x.isnull().sum())
    
    for area_type, jml_missing in missing_per_type.items():
        print(f"  {area_type}: {jml_missing:,} missing values")

# Cek apakah ada negara yang punya banyak missing values di kolom value
# Ini bisa jadi indikasi negara tersebut punya data yang tidak lengkap

if 'area' in df.columns:
    missing_per_negara = df.groupby('area')['value'].apply(lambda x: x.isnull().sum())
    missing_per_negara = missing_per_negara[missing_per_negara > 0].sort_values(ascending=False)
    
    if len(missing_per_negara) > 0:
        print(f"10 Area/Negara dengan missing values terbanyak di kolom 'value':")
        print(missing_per_negara.head(10))
    else:
        print("✅ Tidak ada missing values di kolom 'value'")

In [ ]:
#menghapus baris yang kolom date nya kosong
sebelum = len(df)
df = df.dropna(subset=['date'])
print(f'hapus date kosong : dihapus {sebelum - len(df):,} baris')

#menghapus baris yang kolom value nya kosong
sebelum = len(df)
df = df.dropna(subset=['value'])
print(f'hapus value kosong : dihapus {sebelum - len(df):,} baris')

#kolom yoy dibiarkan saja
print(f'Kolom YoY          : dibiarkan kosong (wajar di awal periode)')

print(f'\nTotal baris setelah dibersihkan: {len(df):,}')

# 7. Hapus data duplikat
- agar tidak terjadi kesalahan pada proses pengumpulan atau penggabungan data
- Agar tidak terjadi double counting saat agregasi 

meski di data ini ternyata tidak ditemukan duplikat, pengecekan tetap wajib dilakukan sebagai validasi

In [ ]:
jumlah_duplikat = df.duplicated().sum()
print(f'Jumlah baris duplikat: {jumlah_duplikat:,}')

if jumlah_duplikat > 0:
    sebelum = len(df)
    df = df.drop_duplicates()
    print(f'Duplikat dihapus: {sebelum - len(df):,} baris dihapus')
else:
    print('Tidak ada duplikat — data sudah bersih!')

# 8. Validasi Struktur Hierarki
- Dataset memiliki 5 Category, 7 Subcategory, dan 5 Unit yang harus konsisten agar analisis tidak salah
- Data 12 bulan terakhir ditandai sebagai provisional karena Ember biasanya merevisi data periode tersebut

In [ ]:
# CEK NILAI UNIK KOLOM HIERARKI

kolom_hierarki = ['category', 'subcategory', 'variable', 'unit', 'area_type']

for kolom in kolom_hierarki:
    if kolom in df.columns:
        nilai_unik = df[kolom].unique()
        print(f"\nKolom '{kolom}' ({len(nilai_unik)} nilai unik):")
        for val in sorted(nilai_unik):
            print(f"   - {val}")

In [ ]:
# CEK DISTRIBUSI AREA TYPE
# Penting! Jangan campur Country dengan Region saat analisis

if 'area_type' in df.columns:
    print("Distribusi Area Type (berapa banyak baris per tipe area):")
    distribusi = df['area_type'].value_counts()
    for area_type, jumlah in distribusi.items():
        persen = (jumlah / len(df)) * 100
        print(f"  {area_type}: {jumlah:,} baris ({persen:.1f}%)")

In [ ]:
# CEK KELENGKAPAN DATA PER NEGARA
# Cek berapa tahun data yang tersedia per negara
# (beberapa negara kecil baru punya data setelah 2018-2020)

if 'area_type' in df.columns and 'area' in df.columns:
    df_negara = df[df['area_type'] == 'Country or economy']
    
    rentang_per_negara = df_negara.groupby('area')['year'].agg(['min', 'max', 'nunique'])
    rentang_per_negara.columns = ['Tahun Mulai', 'Tahun Akhir', 'Jumlah Tahun']
    rentang_per_negara = rentang_per_negara.sort_values('Tahun Mulai', ascending=False)
    
    print("10 negara dengan data PALING BARU (mulai paling telat):")
    print(rentang_per_negara.head(10))
    
    print(f"\nTotal negara/ekonomi dalam dataset: {len(rentang_per_negara)}")

# 9. Penandaan Data Provisional (12 Bulan Terakhir)

In [ ]:
tanggal_terakhir = df['date'].max()
cutoff_provisional = tanggal_terakhir - pd.DateOffset(months=12)

df['is_provisional'] = (df['date'] > cutoff_provisional).astype('int8')

print(f'Tanggal terakhir dataset : {tanggal_terakhir.strftime("%Y-%m")}')
print(f'Cutoff provisional        : setelah {cutoff_provisional.strftime("%Y-%m")}')
print(f'Baris provisional (flag=1): {df["is_provisional"].sum():,}')
print(f'Baris final       (flag=0): {(df["is_provisional"] == 0).sum():,}')
print()
print('Contoh baris provisional:')
df[df['is_provisional'] == 1][['date', 'area', 'category', 'variable', 'value', 'is_provisional']].head()

# 10. Pemisahan Data
Dataset mencampur data Country or economy (445.688 baris) dengan Region (55.795 baris) 
- jika tidak dipisah, data regional yang sudah merupakan agregasi negara-negara akan ikut terhitung dan menyebabkan double counting
- Pemisahan per unit (TWh, %, mtCO2) agar analisis tidak tercampur satuan yang berbeda

In [ ]:
# Memisahkan data negara (Country) dari data agregat (Region/World)
df_country = df[df['area_type'] == 'Country or economy'].copy()
df_region = df[df['area_type'] != 'Country or economy'].copy()

# Memisahkan data berdasarkan satuan (Unit) agar analisis tidak campur aduk
df_twh = df_country[df_country['unit'] == 'TWh'].copy()
df_percent = df_country[df_country['unit'] == '%'].copy()
df_emissions = df_country[df_country['unit'] == 'mtCO2'].copy()

print("Hasil pemisahan data (semua sudah bersih):")
print(f"  df_country  (semua unit) : {df_country.shape[0]:,} baris")
print(f"  df_region   (semua unit) : {df_region.shape[0]:,} baris")
print(f"  df_twh      (TWh saja)   : {df_twh.shape[0]:,} baris")
print(f"  df_percent  (% saja)     : {df_percent.shape[0]:,} baris")
print(f"  df_emissions (mtCO2 saja): {df_emissions.shape[0]:,} baris")

# 11. Analisis Outlier pada Kolom Value

In [ ]:
print('Analisis Outlier per Satuan (df_country):')
print('=' * 55)

for unit in sorted(df_country['unit'].unique()):
    subset = df_country[df_country['unit'] == unit]['value'].dropna()
    Q1 = subset.quantile(0.25)
    Q3 = subset.quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas  = Q3 + 1.5 * IQR
    outliers = subset[(subset < batas_bawah) | (subset > batas_atas)]
    print(f'Satuan : {unit}')
    print(f'  Range nilai   : [{subset.min():.2f}, {subset.max():.2f}]')
    print(f'  Batas IQR     : [{batas_bawah:.2f}, {batas_atas:.2f}]')
    print(f'  Jumlah outlier: {len(outliers):,} ({len(outliers)/len(subset)*100:.1f}%)')
    print()

print('Kesimpulan: Outlier TIDAK dihapus.')
print('Nilai ekstrem (China, USA, India) adalah data valid, bukan error input.')

# Coba Validasi

In [ ]:
# Ambil sampel data Indonesia satuan TWh
df_check = df_country[(df_country['area'] == 'Indonesia') & (df_country['unit'] == 'TWh')]

# 1. Cek: Apakah total bahan bakar == total pembangkitan?
fuel_sum = df_check[df_check['subcategory'] == 'Fuel'].groupby('date')['value'].sum()
total_gen = df_check[df_check['variable'] == 'Total Generation'].set_index('date')['value']

# Hasil Validasi
print(f"Data Match: {np.isclose(fuel_sum, total_gen).all()}")

# 12. Ringkasan Akhir dan Simpan Data
- Agar ada dokumentasi perubahan yang terjadi selama preprocessing
- Agar data original tetap aman karena hasil disimpan ke file baru ember_cleaned.csv

In [29]:
# ===== RINGKASAN PERUBAHAN =====
print("=" * 50)
print("       RINGKASAN HASIL PREPROCESSING")
print("=" * 50)
print(f'Data awal           : {df_asli.shape[0]:>8,} baris')
print(f'Data akhir (bersih) : {df.shape[0]:>8,} baris')
print(f'Baris dihapus       : {df_asli.shape[0] - df.shape[0]:>8,} baris')
print(f'Kolom baru          : year, month, month_name, quarter, yearmonth')
print('-' * 50)
print(f'Duplikat tersisa    : {df.duplicated().sum()}')

yoy_cols = [c for c in df.columns if 'yoy' in c.lower()]
missing_kritis = df.drop(columns=yoy_cols).isnull().sum().sum()
print(f'Missing values kritis (non-YOY): {missing_kritis}')
print('=' * 50)

print()
print("Tipe data akhir:")
print(df.dtypes)

       RINGKASAN HASIL PREPROCESSING
Data awal           :  501,483 baris
Data akhir (bersih) :  498,191 baris
Baris dihapus       :    3,292 baris
Kolom baru          : year, month, month_name, quarter, yearmonth
--------------------------------------------------
Duplikat tersisa    : 0
Missing values kritis (non-YOY): 0

Tipe data akhir:
area                         category
iso_3_code                        str
date                   datetime64[us]
area_type                    category
continent                    category
ember_region                 category
eu                               Int8
oecd                             Int8
g20                              Int8
g7                               Int8
asean                            Int8
category                     category
subcategory                  category
variable                     category
unit                         category
value                         float64
yoy_absolute_change           float64
yoy_pct_chan

In [ ]:
# SIMPAN DATA YANG SUDAH BERSIH
# Data yang sudah dibersihkan disimpan ke file baru

NAMA_FILE_BERSIH = 'ember_cleaned.csv'

df.to_csv(NAMA_FILE_BERSIH, index=False)

print(f"✅ Data bersih berhasil disimpan ke: '{NAMA_FILE_BERSIH}'")
print(f"   Total baris tersimpan: {len(df):,}")